<a href="https://colab.research.google.com/github/scanonc/si3003-workshop-rag-scanonc/blob/main/02b_rag_app_nvidia_nim_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aplicación RAG con NVIDIA NIM en Google Colab

Esta versión es **autosuficiente**: configura NVIDIA NIM, carga los documentos, crea un índice Qdrant temporal y publica una interfaz Gradio.

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`

> En Colab, ejecuta las celdas en orden. La última celda mostrará una URL pública temporal de Gradio.


## 1. Instalar dependencias

Después de la instalación no suele ser necesario reiniciar el runtime. Si Colab lo solicita, reinícialo y vuelve a ejecutar desde la celda siguiente.


In [2]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configurar la API key

En Colab, abre **🔑 Secrets**, crea un secreto llamado `NVIDIA_API_KEY` y habilita el acceso para este notebook. También puedes introducirla de forma oculta cuando la celda la solicite.


In [3]:
import os
import json
import getpass
import time
from pathlib import Path
from typing import List, Dict

import gradio as gr
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
COLLECTION = "company_docs_gradio"
QDRANT_PATH = "qdrant_data"

print("✅ Configuración NVIDIA NIM lista")


✅ Configuración NVIDIA NIM lista


## 3. Cargar los documentos

Si quieres usar documentos propios, súbelos a `data/company_documents.json` con los campos `id`, `title`, `content` y, opcionalmente, `metadata`. Si el archivo no existe, la aplicación usa un conjunto de demostración.


In [19]:
import json
from pathlib import Path
from langchain_core.documents import Document

possible_paths = [
    Path.cwd() / "data" / "company_documents.json",
    Path.cwd().parent / "data" / "company_documents.json",
]

corpus_path = next(
    (path for path in possible_paths if path.is_file()),
    None,
)

if corpus_path is None:
    raise FileNotFoundError(
        f"No se encontró company_documents.json. "
        f"Directorio actual: {Path.cwd()}"
    )

with corpus_path.open("r", encoding="utf-8") as f:
    raw_documents = json.load(f)

documents = [
    Document(
        page_content=item["content"],
        metadata={
            "id": item["id"],
            "title": item["title"],
            "source": item["title"],
            **item["metadata"],
        },
    )
    for item in raw_documents
]

print(f"Documentos cargados: {len(documents)}")
print(f"Ruta utilizada: {corpus_path.resolve()}")

Documentos cargados: 8
Ruta utilizada: C:\Users\sebas\Downloads\si3003-workshop-rag-scanonc\data\company_documents.json


## 4. Construir el pipeline RAG

Esta celda crea los fragmentos, calcula embeddings con NVIDIA NIM y los indexa en Qdrant. El índice vive en memoria mientras la sesión de Colab esté activa.


In [20]:
def build_pipeline(documents: List[Dict], reindex: bool = False):
    embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
    llm = ChatNVIDIA(
        model=GENERATION_MODEL,
        temperature=0.2,
        top_p=0.95,
        max_completion_tokens=700,
    )

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=80,
        separators=["\n\n", "\n", ".", " "],
    )

    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))

    qdrant_client = QdrantClient(path=QDRANT_PATH)
    collection_exists = qdrant_client.collection_exists(COLLECTION)
    if reindex and collection_exists:
        qdrant_client.delete_collection(COLLECTION)
        collection_exists = False
    if not collection_exists:
        # La dimensión se consulta solo al crear la colección.
        embedding_dim = len(embeddings.embed_query("prueba de dimensión"))
        qdrant_client.create_collection(
            collection_name=COLLECTION,
            vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
        )

    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )
    if not collection_exists:
        vectorstore.add_documents(chunks)

    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    )

    prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Eres un asistente empresarial que responde preguntas utilizando
        únicamente el contexto proporcionado.

        Reglas obligatorias:
        1. Responde únicamente con información respaldada por el contexto.
        2. Cita el título del documento que respalda la respuesta entre
        corchetes, por ejemplo: [Política de trabajo remoto].
        3. Si el contexto no contiene información suficiente para responder,
        responde exactamente:
        "No tengo información suficiente en los documentos para responder."
        4. No inventes datos ni completes información con conocimientos externos.
        5. Responde en el mismo idioma de la pregunta.
        6. Responde en un máximo de 3 oraciones.
        7. Si la pregunta es ambigua y el contexto contiene varias opciones,
        indica la ambigüedad en lugar de asumir una respuesta.
        """
    ),
    (
        "human",
        """
        CONTEXTO:
        {context}

        PREGUNTA:
        {question}
        """
    ),
])

    def format_docs(items: List[Document]) -> str:
        return "\n\n---\n\n".join(
            f"[{item.metadata.get('title', 'Sin título')}]\n{item.page_content}"
            for item in items
        )

    setup_and_retrieval = RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    pipeline = setup_and_retrieval.assign(answer=answer_from_documents)
    return pipeline, chunks, vectorstore


print("Construyendo el índice...")
rag_pipeline, indexed_chunks, vectorstore = build_pipeline(company_documents)
titles = [document["title"] for document in company_documents]
print(f"✅ Pipeline listo: {len(titles)} documentos, {len(indexed_chunks)} chunks")


Construyendo el índice...


c:\Users\sebas\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ Pipeline listo: 8 documentos, 8 chunks


## Persistencia local y tiempos de indexación

La primera ejecución fuerza la indexación para medir su costo. La segunda solo abre la colección persistida y verifica que puede responder sin volver a insertar embeddings. Para simular un reinicio, ejecuta esta celda de nuevo después de reiniciar el kernel.


In [ ]:
index_start = time.perf_counter()
indexed_pipeline, indexed_chunks, indexed_vectorstore = build_pipeline(
    company_documents,
    reindex=True,
)
index_seconds = time.perf_counter() - index_start

reopen_start = time.perf_counter()
rag_pipeline, indexed_chunks, vectorstore = build_pipeline(
    company_documents,
    reindex=False,
)
reopen_seconds = time.perf_counter() - reopen_start

test_result = rag_pipeline.invoke("¿Cuántos días por semana pueden trabajar remotamente los empleados?")
print("| Operación | Tiempo (s) | Resultado |")
print("|---|---:|---|")
print(f"| Indexar y persistir embeddings | {index_seconds:.2f} | {len(indexed_chunks)} chunks |")
print(f"| Reabrir colección existente | {reopen_seconds:.2f} | {len(test_result['documents'])} documentos recuperados |")
print(f"\nRuta persistente: {QDRANT_PATH}")


## C5 — MMR frente a similitud pura

Esta celda compara los cinco chunks recuperados por similitud pura y por MMR. La cobertura se mide como el número de títulos distintos y la redundancia como los chunks que repiten un título ya visto.


In [ ]:
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5},
)

comparison_questions = [
    "¿Cuántos días por semana pueden trabajar remotamente los empleados?",
    "¿Qué deben completar los nuevos empleados durante la primera semana?",
    "¿Qué normas existen sobre seguridad y reporte de incidentes?",
]

def print_retrieval_strategy(name, retrieved_documents):
    titles = [
        document.metadata.get("title", "Sin título")
        for document in retrieved_documents
    ]
    unique_titles = set(titles)
    redundancy = len(titles) - len(unique_titles)
    print(f"\n{name}")
    print(f"Cobertura: {len(unique_titles)} títulos distintos de {len(titles)} chunks")
    print(f"Redundancia: {redundancy} chunks repetidos")
    for index, document in enumerate(retrieved_documents, start=1):
        title = document.metadata.get("title", "Sin título")
        print(f"{index}. [{title}] {document.page_content}")

for question in comparison_questions:
    print(f"\n{'=' * 80}\nPREGUNTA: {question}\n{'=' * 80}")
    print_retrieval_strategy("SIMILITUD PURA", similarity_retriever.invoke(question))
    print_retrieval_strategy("MMR (lambda_mult=0.5)", mmr_retriever.invoke(question))


## 5. Crear la interfaz Gradio

In [22]:
def chat(question: str, history: list | None):
    history = history or []
    question = (question or "").strip()
    if not question:
        return "", history, "Escribe una pregunta."

    try:
        result = rag_pipeline.invoke(question)
        answer = result["answer"].strip()
        documents = result["documents"]
        scored_documents = vectorstore.similarity_search_with_score(question, k=3)

        unique_sources = []
        for document in documents:
            title = document.metadata.get("title", "Sin título")
            if title not in unique_sources:
                unique_sources.append(title)

        if scored_documents:
            sources_md = "**Fuentes recuperadas (score de similitud):**\n" + "\n".join(
                f"- {document.metadata.get('title', 'Sin título')} — `{score:.4f}`"
                for document, score in scored_documents
            )
        else:
            sources_md = "No se encontraron documentos relevantes."

    except Exception as error:
        answer = "Ocurrió un error al consultar el pipeline. Revisa el detalle mostrado debajo."
        sources_md = f"**Error:** `{type(error).__name__}: {error}`"

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return "", history, sources_md


def clear_chat():
    return [], ""


with gr.Blocks(title="RAG empresarial con NVIDIA NIM") as demo:
    gr.Markdown(
        "# RAG empresarial\n"
        "Consulta documentos internos mediante **LangChain + NVIDIA NIM + Qdrant**."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Documentos indexados")
            gr.Markdown("\n".join(f"- {title}" for title in titles))
            gr.Markdown(f"*{len(indexed_chunks)} chunks en el índice*")
            gr.Markdown("---")
            gr.Markdown(
                f"**Embeddings:** `{EMBEDDING_MODEL}`  \n"
                f"**LLM:** `{GENERATION_MODEL}`  \n"
                "**Vector store:** Qdrant en memoria  \n"
                "**Retrieval:** k=3, threshold=0.3"
            )

        with gr.Column(scale=2):
            # Gradio 6 usa mensajes {role, content} por defecto;
            # el antiguo argumento type="messages" ya no existe.
            chatbot = gr.Chatbot(height=430, label="Chat")
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Haz una pregunta sobre los documentos...",
                    label="Pregunta",
                    lines=2,
                    scale=4,
                )
                send_button = gr.Button("Enviar", variant="primary", scale=1)

            sources_box = gr.Markdown()
            clear_button = gr.Button("Limpiar", variant="secondary")

    gr.Examples(
        examples=[
            ["¿Cuál es el precio del plan Enterprise de CloudSync Pro?"],
            ["¿Cuántos días pueden trabajar remotamente los empleados?"],
            ["¿Qué pasa con los reembolsos mayores a $100?"],
            ["¿Qué ocurre durante la primera semana de onboarding?"],
        ],
        inputs=question_box,
        label="Preguntas de ejemplo",
    )

    send_button.click(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    question_box.submit(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    clear_button.click(clear_chat, outputs=[chatbot, sources_box])


## 6. Ejecutar la aplicación en Colab

`share=True` crea el enlace público temporal. La aplicación estará disponible mientras esta celda y el runtime de Colab continúen activos.


In [ ]:
demo.queue().launch(
    share=True,
    debug=True,
    show_error=True,
)


### Detener o volver a ejecutar

- Interrumpe la última celda para detener el servidor.
- Antes de volver a ejecutarla, puedes cerrar instancias previas con `gr.close_all()`.
- El enlace público cambia cada vez que se inicia la aplicación.


In [25]:
# Ejecuta esta celda solo cuando quieras cerrar servidores Gradio anteriores.
gr.close_all()
